# CLIP Vision Fine-Tuning on Artificio/WikiArt (137 Styles)

This notebook implements:
1. Random guessing baseline
2. Standard CLIP zero-shot baseline
3. Linear probe baseline
4. Fine-tuned CLIP (vision encoder only, staged unfreezing)


In [1]:
from __future__ import annotations

import copy
import dataclasses
import hashlib
import io
import json
import math
import os
import random
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, f1_score
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision import transforms
from tqdm.auto import tqdm

from datasets import ClassLabel, DatasetDict, Features, Image as HFImage, Value, concatenate_datasets, load_dataset
from transformers import CLIPImageProcessor, CLIPModel, CLIPProcessor

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


Torch: 2.10.0+cu128
CUDA available: True


In [ ]:
@dataclass
class TrainingConfig:
    dataset_id: str = "Artificio/WikiArt"
    model_id: str = "openai/clip-vit-base-patch32"
    num_labels: int = 137
    seed: int = 42

    output_dir: str = "./artifacts_clip_wikiart"
    checkpoint_dir: str = "./artifacts_clip_wikiart/checkpoints"

    train_batch_size_per_device: int = 32
    eval_batch_size_per_device: int = 64
    grad_accum_steps: int = 1
    num_workers: int = 8
    pin_memory: bool = True

    head_only_epochs: int = 2
    partial_unfreeze_epochs: int = 8
    unfreeze_last_n_blocks: int = 4

    head_lr: float = 5e-4
    vision_lr: float = 1e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.10
    max_grad_norm: float = 1.0

    dropout: float = 0.1
    label_smoothing: float = 0.0
    use_class_weights: bool = True

    early_stopping_patience: int = 3
    eval_every_epoch: bool = True

    smoke_subset_size: int = 1000
    smoke_epochs: int = 1

    zero_shot_prompt_template: str = "a painting in the {} style"

    save_confusion_matrix: bool = True

    distributed_backend: str = "nccl"
    force_fp16_if_no_bf16: bool = True


config = TrainingConfig()
print(config)


TrainingConfig(dataset_id='Artificio/WikiArt', model_id='openai/clip-vit-base-patch32', num_labels=137, seed=42, output_dir='./artifacts_clip_wikiart', checkpoint_dir='./artifacts_clip_wikiart/checkpoints', train_batch_size_per_device=256, eval_batch_size_per_device=1024, grad_accum_steps=1, num_workers=8, pin_memory=True, head_only_epochs=2, partial_unfreeze_epochs=8, unfreeze_last_n_blocks=4, head_lr=0.0005, vision_lr=1e-05, weight_decay=0.01, warmup_ratio=0.1, max_grad_norm=1.0, dropout=0.1, label_smoothing=0.0, use_class_weights=True, early_stopping_patience=3, eval_every_epoch=True, smoke_subset_size=1000, smoke_epochs=1, zero_shot_prompt_template='a painting in the {} style', save_confusion_matrix=True, distributed_backend='nccl', force_fp16_if_no_bf16=True)


In [3]:
def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_dist_info() -> Tuple[int, int, int]:
    rank = int(os.environ.get("RANK", "0"))
    world_size = int(os.environ.get("WORLD_SIZE", "1"))
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    return rank, world_size, local_rank


def is_main_process() -> bool:
    rank, _, _ = get_dist_info()
    return rank == 0


def maybe_init_distributed(cfg: TrainingConfig) -> None:
    rank, world_size, _ = get_dist_info()
    if world_size > 1 and not dist.is_initialized():
        dist.init_process_group(backend=cfg.distributed_backend, rank=rank, world_size=world_size)


def maybe_destroy_distributed() -> None:
    if dist.is_initialized():
        dist.destroy_process_group()


def get_device() -> torch.device:
    _, world_size, local_rank = get_dist_info()
    if torch.cuda.is_available():
        if world_size > 1:
            torch.cuda.set_device(local_rank)
            return torch.device(f"cuda:{local_rank}")
        return torch.device("cuda")
    return torch.device("cpu")


def get_autocast_dtype(cfg: TrainingConfig) -> Optional[torch.dtype]:
    if not torch.cuda.is_available():
        return None
    if torch.cuda.is_bf16_supported():
        return torch.bfloat16
    if cfg.force_fp16_if_no_bf16:
        return torch.float16
    return None


def unwrap_model(model: nn.Module) -> nn.Module:
    return model.module if isinstance(model, DDP) else model


def ensure_output_dirs(cfg: TrainingConfig) -> None:
    Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
    Path(cfg.checkpoint_dir).mkdir(parents=True, exist_ok=True)


set_global_seed(config.seed)
ensure_output_dirs(config)


In [4]:
def _infer_label_columns(ds) -> Tuple[str, Optional[str]]:
    cols = set(ds.column_names)
    numeric_candidates = ["label", "labels", "style_id", "target"]
    text_candidates = ["style", "style_name", "genre", "label_name", "class_name"]

    label_col = None
    for c in numeric_candidates:
        if c in cols:
            label_col = c
            break

    text_col = None
    for c in text_candidates:
        if c in cols:
            text_col = c
            break

    if label_col is None and text_col is None:
        raise ValueError(f"Could not infer label column from columns: {sorted(cols)}")

    return label_col if label_col is not None else text_col, text_col


def _infer_image_column(ds) -> str:
    for c in ds.column_names:
        feat = ds.features[c]
        if isinstance(feat, HFImage):
            return c
    for c in ["image", "img", "jpg", "png"]:
        if c in ds.column_names:
            return c
    raise ValueError("Could not infer image column.")


def _build_label_map_from_text(train_ds, text_col: str, expected_num_labels: int) -> Dict[str, int]:
    unique_styles = sorted(set(train_ds[text_col]))
    if len(unique_styles) != expected_num_labels:
        raise ValueError(
            f"Expected {expected_num_labels} unique styles but found {len(unique_styles)} in '{text_col}'."
        )
    return {name: i for i, name in enumerate(unique_styles)}


def _canonicalize_dataset(ds, image_col: str, label_col: str, text_col: Optional[str], label_name_to_id: Optional[Dict[str, int]] = None):
    def _row_map(ex):
        out = {
            "image": ex[image_col],
        }
        if text_col is not None and label_name_to_id is not None:
            style_name = ex[text_col]
            out["label"] = int(label_name_to_id[style_name])
            out["style_name"] = style_name
        else:
            out["label"] = int(ex[label_col])
            out["style_name"] = str(ex[label_col])
        img_obj = ex[image_col]
        if isinstance(img_obj, dict) and "path" in img_obj and img_obj["path"]:
            raw_id = str(img_obj["path"])
        else:
            try:
                b = img_obj.tobytes()
                raw_id = hashlib.md5(b).hexdigest()
            except Exception:
                raw_id = str(id(img_obj))
        out["image_id"] = raw_id
        return out

    mapped = ds.map(_row_map, remove_columns=ds.column_names)
    return mapped


def _try_stratified_split(ds, test_size: float, seed: int):
    try:
        return ds.train_test_split(test_size=test_size, seed=seed, stratify_by_column="label")
    except Exception as e:
        print(f"Warning: stratified split failed ({e}). Falling back to random split.")
        return ds.train_test_split(test_size=test_size, seed=seed)


def _split_train_val_test(ds, seed: int) -> DatasetDict:
    tt = _try_stratified_split(ds, test_size=0.2, seed=seed)
    train = tt["train"]
    holdout = tt["test"]
    vt = _try_stratified_split(holdout, test_size=0.5, seed=seed)
    return DatasetDict(train=train, validation=vt["train"], test=vt["test"])


def _drop_duplicates_by_image_id(ds):
    seen = set()
    keep_idx = []
    for i, img_id in enumerate(ds["image_id"]):
        if img_id in seen:
            continue
        seen.add(img_id)
        keep_idx.append(i)
    if len(keep_idx) == len(ds):
        return ds
    return ds.select(keep_idx)


def _deduplicate_datasetdict_by_image_id(dsd: DatasetDict) -> DatasetDict:
    split_order = [k for k in ["train", "validation", "test"] if k in dsd]
    split_order += [k for k in dsd.keys() if k not in split_order]

    seen = set()
    out = {}
    removed = {}

    for split in split_order:
        ds = dsd[split]
        keep_idx = []
        dropped = 0
        for i, img_id in enumerate(ds["image_id"]):
            if img_id in seen:
                dropped += 1
                continue
            seen.add(img_id)
            keep_idx.append(i)
        out[split] = ds.select(keep_idx) if len(keep_idx) != len(ds) else ds
        removed[split] = dropped

    total_removed = sum(removed.values())
    if total_removed > 0:
        print(f"Deduplicated by image_id before splitting/checks. Removed: {removed}")
    return DatasetDict(out)


def _ensure_train_has_all_labels(dataset_dict: DatasetDict, num_labels: int, seed: int) -> DatasetDict:
    train = dataset_dict["train"]
    val = dataset_dict["validation"]
    test = dataset_dict["test"]

    expected = set(range(num_labels))
    train_labels = set(train["label"])
    missing = sorted(expected.difference(train_labels))
    if not missing:
        return dataset_dict

    val_labels = np.array(val["label"])
    test_labels = np.array(test["label"])
    move_val_idx = []
    move_test_idx = []
    unresolved = []

    for lbl in missing:
        val_hits = np.where(val_labels == lbl)[0]
        if len(val_hits) > 0:
            move_val_idx.append(int(val_hits[0]))
            continue
        test_hits = np.where(test_labels == lbl)[0]
        if len(test_hits) > 0:
            move_test_idx.append(int(test_hits[0]))
            continue
        unresolved.append(lbl)

    if unresolved:
        raise ValueError(
            f"Missing labels are absent from all splits after mapping: {unresolved}."
        )

    moved_parts = []
    if move_val_idx:
        moved_parts.append(val.select(move_val_idx))
        keep_val = sorted(set(range(len(val))) - set(move_val_idx))
        val = val.select(keep_val)
    if move_test_idx:
        moved_parts.append(test.select(move_test_idx))
        keep_test = sorted(set(range(len(test))) - set(move_test_idx))
        test = test.select(keep_test)

    train = concatenate_datasets([train] + moved_parts).shuffle(seed=seed)
    print(
        f"Adjusted split to keep all labels in train. Moved {len(move_val_idx)} from validation and {len(move_test_idx)} from test."
    )
    return DatasetDict(train=train, validation=val, test=test)


def _class_distribution_report(dataset_dict: DatasetDict) -> Dict[str, Dict[int, int]]:
    report = {}
    for split in ["train", "validation", "test"]:
        vals, counts = np.unique(np.array(dataset_dict[split]["label"]), return_counts=True)
        report[split] = {int(v): int(c) for v, c in zip(vals, counts)}
    return report


def _check_split_leakage(dataset_dict: DatasetDict) -> None:
    train_ids = set(dataset_dict["train"]["image_id"])
    val_ids = set(dataset_dict["validation"]["image_id"])
    test_ids = set(dataset_dict["test"]["image_id"])

    train_val_overlap = train_ids.intersection(val_ids)
    train_test_overlap = train_ids.intersection(test_ids)
    val_test_overlap = val_ids.intersection(test_ids)

    assert len(train_val_overlap) == 0, f"Leakage train/val: {len(train_val_overlap)}"
    assert len(train_test_overlap) == 0, f"Leakage train/test: {len(train_test_overlap)}"
    assert len(val_test_overlap) == 0, f"Leakage val/test: {len(val_test_overlap)}"


def prepare_dataset(cfg: TrainingConfig) -> DatasetDict:
    raw = load_dataset(cfg.dataset_id)
    sample_split = "train" if "train" in raw else list(raw.keys())[0]
    sample_ds = raw[sample_split]

    image_col = _infer_image_column(sample_ds)
    label_col, text_col = _infer_label_columns(sample_ds)

    label_name_to_id = None
    if text_col is not None:
        base_train_for_map = raw["train"] if "train" in raw else sample_ds
        label_name_to_id = _build_label_map_from_text(base_train_for_map, text_col, cfg.num_labels)

    canonical = {}
    for split_name, split_ds in raw.items():
        canonical[split_name] = _canonicalize_dataset(split_ds, image_col, label_col, text_col, label_name_to_id)
    canonical = DatasetDict(canonical)
    canonical = _deduplicate_datasetdict_by_image_id(canonical)

    if all(k in canonical for k in ["train", "validation", "test"]):
        dataset_dict = DatasetDict(
            train=canonical["train"],
            validation=canonical["validation"],
            test=canonical["test"],
        )
    elif all(k in canonical for k in ["train", "test"]):
        tv = _try_stratified_split(canonical["train"], test_size=0.111111, seed=cfg.seed)
        dataset_dict = DatasetDict(train=tv["train"], validation=tv["test"], test=canonical["test"])
    else:
        first = canonical[list(canonical.keys())[0]]
        dataset_dict = _split_train_val_test(first, seed=cfg.seed)

    dataset_dict = _ensure_train_has_all_labels(dataset_dict, cfg.num_labels, cfg.seed)

    unique_train = sorted(set(dataset_dict["train"]["label"]))
    if len(unique_train) != cfg.num_labels:
        raise ValueError(
            f"Expected {cfg.num_labels} labels in train split, found {len(unique_train)}."
        )

    names = [None] * cfg.num_labels
    if label_name_to_id is not None:
        for n, i in label_name_to_id.items():
            names[i] = n
    else:
        names = [f"style_{i}" for i in range(cfg.num_labels)]

    for i, n in enumerate(names):
        if n is None:
            names[i] = f"style_{i}"

    label_feature = ClassLabel(num_classes=cfg.num_labels, names=names)
    target_features = Features({
        "image": HFImage(),
        "label": label_feature,
        "style_name": Value("string"),
        "image_id": Value("string"),
    })
    dataset_dict = dataset_dict.cast(target_features)

    _check_split_leakage(dataset_dict)
    dist_report = _class_distribution_report(dataset_dict)

    meta = {
        "dataset_id": cfg.dataset_id,
        "num_labels": cfg.num_labels,
        "id2label": {str(i): name for i, name in enumerate(names)},
        "label2id": {name: i for i, name in enumerate(names)},
        "class_distribution": dist_report,
        "split_sizes": {k: len(v) for k, v in dataset_dict.items()},
    }

    if is_main_process():
        with open(Path(cfg.output_dir) / "label_map.json", "w", encoding="utf-8") as f:
            json.dump({"id2label": meta["id2label"], "label2id": meta["label2id"]}, f, indent=2)
        with open(Path(cfg.output_dir) / "dataset_report.json", "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2)

    print("Prepared dataset with split sizes:", meta["split_sizes"])
    return dataset_dict


In [5]:
def get_transforms(processor: CLIPImageProcessor):
    image_mean = processor.image_mean
    image_std = processor.image_std
    size = processor.crop_size["height"] if isinstance(processor.crop_size, dict) else processor.crop_size

    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(size, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=image_mean, std=image_std),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize(size + 32),
        transforms.CenterCrop(size),
        transforms.ToTensor(),
        transforms.Normalize(mean=image_mean, std=image_std),
    ])
    return train_tf, eval_tf


def build_torch_splits(dataset_dict: DatasetDict, cfg: TrainingConfig):
    processor = CLIPImageProcessor.from_pretrained(cfg.model_id)
    train_tf, eval_tf = get_transforms(processor)

    def train_transform(batch):
        images = [img.convert("RGB") if isinstance(img, Image.Image) else img["image"].convert("RGB") for img in batch["image"]]
        batch["pixel_values"] = [train_tf(im) for im in images]
        batch["labels"] = batch["label"]
        return batch

    def eval_transform(batch):
        images = [img.convert("RGB") if isinstance(img, Image.Image) else img["image"].convert("RGB") for img in batch["image"]]
        batch["pixel_values"] = [eval_tf(im) for im in images]
        batch["labels"] = batch["label"]
        return batch

    train_ds = dataset_dict["train"].with_transform(train_transform)
    val_ds = dataset_dict["validation"].with_transform(eval_transform)
    test_ds = dataset_dict["test"].with_transform(eval_transform)
    return processor, train_ds, val_ds, test_ds


def collate_fn(batch):
    pixel_values = torch.stack([x["pixel_values"] for x in batch])
    labels = torch.tensor([int(x["labels"]) for x in batch], dtype=torch.long)
    return {"pixel_values": pixel_values, "labels": labels}


In [6]:
def clip_image_features_compat(clip_model: CLIPModel, pixel_values: torch.Tensor) -> torch.Tensor:
    out = clip_model.get_image_features(pixel_values=pixel_values)
    if isinstance(out, torch.Tensor):
        return out
    if hasattr(out, "image_embeds") and out.image_embeds is not None:
        return out.image_embeds
    if hasattr(out, "pooler_output") and out.pooler_output is not None:
        return out.pooler_output
    if hasattr(out, "last_hidden_state") and out.last_hidden_state is not None:
        return out.last_hidden_state[:, 0, :]
    raise TypeError(f"Unsupported get_image_features output type: {type(out)}")


def clip_text_features_compat(clip_model: CLIPModel, **text_inputs) -> torch.Tensor:
    out = clip_model.get_text_features(**text_inputs)
    if isinstance(out, torch.Tensor):
        return out
    if hasattr(out, "text_embeds") and out.text_embeds is not None:
        return out.text_embeds
    if hasattr(out, "pooler_output") and out.pooler_output is not None:
        return out.pooler_output
    if hasattr(out, "last_hidden_state") and out.last_hidden_state is not None:
        return out.last_hidden_state[:, 0, :]
    raise TypeError(f"Unsupported get_text_features output type: {type(out)}")


class CLIPStyleClassifier(nn.Module):
    def __init__(self, cfg: TrainingConfig, class_weights: Optional[torch.Tensor] = None):
        super().__init__()
        self.cfg = cfg
        self.clip = CLIPModel.from_pretrained(cfg.model_id)
        d = self.clip.config.projection_dim
        self.dropout = nn.Dropout(cfg.dropout)
        self.head = nn.Linear(d, cfg.num_labels)
        self.class_weights = class_weights

    def forward(self, pixel_values: torch.Tensor, labels: Optional[torch.Tensor] = None):
        image_features = clip_image_features_compat(self.clip, pixel_values)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        logits = self.head(self.dropout(image_features))

        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits,
                labels,
                weight=self.class_weights.to(logits.device) if self.class_weights is not None else None,
                label_smoothing=self.cfg.label_smoothing,
            )
        return {"loss": loss, "logits": logits}


def set_trainable_layers(model: nn.Module, stage: str, n_last_blocks: int) -> None:
    m = unwrap_model(model)
    for p in m.parameters():
        p.requires_grad = False

    for p in m.head.parameters():
        p.requires_grad = True
    for p in m.dropout.parameters():
        p.requires_grad = True

    for p in m.clip.text_model.parameters():
        p.requires_grad = False
    if hasattr(m.clip, "text_projection"):
        m.clip.text_projection.requires_grad = False

    if stage == "head":
        return
    if stage != "partial":
        raise ValueError(f"Unknown stage: {stage}")

    layers = m.clip.vision_model.encoder.layers
    n = len(layers)
    k = min(n_last_blocks, n)
    for block in layers[n-k:]:
        for p in block.parameters():
            p.requires_grad = True
    for p in m.clip.vision_model.post_layernorm.parameters():
        p.requires_grad = True


def count_trainable_params(model: nn.Module) -> int:
    m = unwrap_model(model)
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


def build_model(cfg: TrainingConfig, dataset_dict: DatasetDict) -> nn.Module:
    class_weights = None
    if cfg.use_class_weights:
        y = np.array(dataset_dict["train"]["label"], dtype=np.int64)
        counts = np.bincount(y, minlength=cfg.num_labels)
        inv = 1.0 / np.clip(counts, 1, None)
        w = inv / inv.sum() * cfg.num_labels
        class_weights = torch.tensor(w, dtype=torch.float32)
    return CLIPStyleClassifier(cfg, class_weights=class_weights)


In [7]:
def _split_params_for_weight_decay(named_params):
    no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight", "bn.weight"]
    decay_params, nodecay_params = [], []
    for n, p in named_params:
        if not p.requires_grad:
            continue
        if any(nd in n for nd in no_decay):
            nodecay_params.append(p)
        else:
            decay_params.append(p)
    return decay_params, nodecay_params


def build_optimizer_and_scheduler(model: nn.Module, cfg: TrainingConfig, stage: str, steps_per_epoch: int, epochs: int):
    m = unwrap_model(model)

    head_named = list(m.head.named_parameters(prefix="head"))
    vision_named = []
    for n, p in m.clip.vision_model.named_parameters(prefix="clip.vision_model"):
        if p.requires_grad:
            vision_named.append((n, p))

    head_decay, head_no_decay = _split_params_for_weight_decay(head_named)
    vis_decay, vis_no_decay = _split_params_for_weight_decay(vision_named)

    groups = []
    if len(head_decay) > 0:
        groups.append({"params": head_decay, "lr": cfg.head_lr, "weight_decay": cfg.weight_decay})
    if len(head_no_decay) > 0:
        groups.append({"params": head_no_decay, "lr": cfg.head_lr, "weight_decay": 0.0})

    stage_vision_lr = 0.0 if stage == "head" else cfg.vision_lr
    if len(vis_decay) > 0:
        groups.append({"params": vis_decay, "lr": stage_vision_lr, "weight_decay": cfg.weight_decay})
    if len(vis_no_decay) > 0:
        groups.append({"params": vis_no_decay, "lr": stage_vision_lr, "weight_decay": 0.0})

    optimizer = torch.optim.AdamW(groups)
    total_steps = max(1, math.ceil(steps_per_epoch / cfg.grad_accum_steps) * epochs)
    warmup_steps = int(total_steps * cfg.warmup_ratio)

    def lr_lambda(current_step: int):
        if current_step < warmup_steps:
            return float(current_step) / float(max(1, warmup_steps))
        progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler


def compute_metrics_from_logits(logits: np.ndarray, labels: np.ndarray) -> Dict[str, float]:
    preds = logits.argmax(axis=1)
    top1 = float((preds == labels).mean())

    k = min(5, logits.shape[1])
    topk_idx = np.argpartition(logits, -k, axis=1)[:, -k:]
    top5 = float(np.mean([labels[i] in topk_idx[i] for i in range(len(labels))]))

    macro_f1 = float(f1_score(labels, preds, average="macro"))
    bal_acc = float(balanced_accuracy_score(labels, preds))
    return {
        "top1": top1,
        "top5": top5,
        "macro_f1": macro_f1,
        "balanced_acc": bal_acc,
    }


def evaluate_model(model: nn.Module, loader: DataLoader, device: torch.device, amp_dtype: Optional[torch.dtype]):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            pixel_values = batch["pixel_values"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=(amp_dtype is not None and device.type == "cuda")):
                out = model(pixel_values=pixel_values, labels=None)
            logits = out["logits"].detach().float().cpu().numpy()
            all_logits.append(logits)
            all_labels.append(labels.cpu().numpy())

    logits_np = np.concatenate(all_logits, axis=0)
    labels_np = np.concatenate(all_labels, axis=0)
    metrics = compute_metrics_from_logits(logits_np, labels_np)
    return metrics, logits_np, labels_np


In [8]:
def _make_dataloader(ds, cfg: TrainingConfig, split: str, world_size: int, rank: int):
    is_train = split == "train"
    sampler = None
    if world_size > 1 and is_train:
        sampler = DistributedSampler(ds, num_replicas=world_size, rank=rank, shuffle=True, drop_last=False)

    return DataLoader(
        ds,
        batch_size=cfg.train_batch_size_per_device if is_train else cfg.eval_batch_size_per_device,
        shuffle=(sampler is None and is_train),
        sampler=sampler,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        drop_last=False,
        collate_fn=collate_fn,
    )


def _train_one_epoch(model, loader, optimizer, scheduler, device, cfg, amp_dtype, epoch_seed):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(loader, disable=not is_main_process(), desc="train")
    for step, batch in enumerate(pbar):
        pixel_values = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=(amp_dtype is not None and device.type == "cuda")):
            out = model(pixel_values=pixel_values, labels=labels)
            loss = out["loss"] / cfg.grad_accum_steps

        loss.backward()
        if (step + 1) % cfg.grad_accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * cfg.grad_accum_steps
        if is_main_process():
            pbar.set_postfix(loss=f"{running_loss / (step + 1):.4f}")

    return running_loss / max(1, len(loader))


def _save_checkpoint(cfg: TrainingConfig, model: nn.Module, epoch_idx: int, metrics: Dict[str, float], best_path: Path) -> None:
    state = {
        "epoch": epoch_idx,
        "model_state_dict": unwrap_model(model).state_dict(),
        "metrics": metrics,
        "config": asdict(cfg),
    }
    torch.save(state, best_path)


def _load_checkpoint(path: Path, model: nn.Module, map_location="cpu") -> Dict[str, Any]:
    ckpt = torch.load(path, map_location=map_location)
    unwrap_model(model).load_state_dict(ckpt["model_state_dict"])
    return ckpt


def run_model_wiring_checks(model: nn.Module, cfg: TrainingConfig) -> None:
    base = unwrap_model(model)
    set_trainable_layers(base, stage="head", n_last_blocks=cfg.unfreeze_last_n_blocks)
    vision_trainable_head = sum(p.requires_grad for p in base.clip.vision_model.parameters())
    assert vision_trainable_head == 0, "Stage head should have 0 trainable vision params"

    set_trainable_layers(base, stage="partial", n_last_blocks=cfg.unfreeze_last_n_blocks)
    layers = base.clip.vision_model.encoder.layers
    n = len(layers)
    for i, blk in enumerate(layers):
        blk_trainable = any(p.requires_grad for p in blk.parameters())
        if i < n - cfg.unfreeze_last_n_blocks:
            assert not blk_trainable, f"Block {i} should stay frozen"
        else:
            assert blk_trainable, f"Block {i} should be trainable"

    assert all(not p.requires_grad for p in base.clip.text_model.parameters()), "Text tower must remain frozen"


def train_finetune(cfg: TrainingConfig, dataset_dict: DatasetDict) -> str:
    maybe_init_distributed(cfg)
    rank, world_size, _ = get_dist_info()
    device = get_device()
    amp_dtype = get_autocast_dtype(cfg)

    processor, train_ds, val_ds, test_ds = build_torch_splits(dataset_dict, cfg)
    train_loader = _make_dataloader(train_ds, cfg, "train", world_size, rank)
    val_loader = _make_dataloader(val_ds, cfg, "validation", 1, 0)
    test_loader = _make_dataloader(test_ds, cfg, "test", 1, 0)

    model = build_model(cfg, dataset_dict).to(device)
    if world_size > 1:
        model = DDP(model, device_ids=[device.index] if device.type == "cuda" else None)

    run_model_wiring_checks(model, cfg)

    total_epochs = cfg.head_only_epochs + cfg.partial_unfreeze_epochs
    stage_plan = [("head", cfg.head_only_epochs), ("partial", cfg.partial_unfreeze_epochs)]
    global_epoch = 0

    best_macro_f1 = -1.0
    best_ckpt_path = Path(cfg.checkpoint_dir) / "best_finetuned.pt"
    early_stop_counter = 0
    history = []

    for stage_name, stage_epochs in stage_plan:
        set_trainable_layers(model, stage=stage_name, n_last_blocks=cfg.unfreeze_last_n_blocks)
        if is_main_process():
            print(f"Stage: {stage_name} | trainable params: {count_trainable_params(model):,}")

        optimizer, scheduler = build_optimizer_and_scheduler(
            model, cfg, stage=stage_name, steps_per_epoch=len(train_loader), epochs=stage_epochs
        )

        for _ in range(stage_epochs):
            global_epoch += 1
            if world_size > 1 and isinstance(train_loader.sampler, DistributedSampler):
                train_loader.sampler.set_epoch(cfg.seed + global_epoch)

            train_loss = _train_one_epoch(model, train_loader, optimizer, scheduler, device, cfg, amp_dtype, cfg.seed + global_epoch)

            val_macro = torch.tensor(-1.0, device=device)
            val_metrics = None
            if rank == 0:
                val_metrics, _, _ = evaluate_model(model, val_loader, device, amp_dtype)
                val_macro = torch.tensor(val_metrics["macro_f1"], device=device)

            if world_size > 1:
                dist.broadcast(val_macro, src=0)
            current_macro = float(val_macro.item())

            if rank == 0:
                row = {
                    "epoch": global_epoch,
                    "stage": stage_name,
                    "train_loss": float(train_loss),
                    "val_macro_f1": current_macro,
                }
                history.append(row)
                print(row)

                if current_macro > best_macro_f1:
                    best_macro_f1 = current_macro
                    early_stop_counter = 0
                    _save_checkpoint(cfg, model, global_epoch, val_metrics, best_ckpt_path)
                else:
                    early_stop_counter += 1

            stop_now = torch.tensor(0, device=device)
            if rank == 0 and early_stop_counter >= cfg.early_stopping_patience:
                stop_now = torch.tensor(1, device=device)
            if world_size > 1:
                dist.broadcast(stop_now, src=0)
            if int(stop_now.item()) == 1:
                if rank == 0:
                    print("Early stopping triggered.")
                break

    if rank == 0:
        _ = _load_checkpoint(best_ckpt_path, model, map_location=device)
        test_metrics, logits_np, labels_np = evaluate_model(model, test_loader, device, amp_dtype)
        with open(Path(cfg.output_dir) / "finetune_history.json", "w", encoding="utf-8") as f:
            json.dump(history, f, indent=2)
        with open(Path(cfg.output_dir) / "finetune_test_metrics.json", "w", encoding="utf-8") as f:
            json.dump(test_metrics, f, indent=2)
        if cfg.save_confusion_matrix:
            cm = confusion_matrix(labels_np, logits_np.argmax(axis=1), labels=np.arange(cfg.num_labels))
            cm_path = Path(cfg.output_dir) / "confusion_matrix_finetune.npy"
            np.save(cm_path, cm)
        print("Fine-tuned test metrics:", test_metrics)

    if world_size > 1:
        dist.barrier()
    return str(best_ckpt_path)


In [9]:
def _get_style_names(dataset_dict: DatasetDict, cfg: TrainingConfig) -> List[str]:
    feat = dataset_dict["train"].features["label"]
    if isinstance(feat, ClassLabel):
        names = feat.names
        if len(names) == cfg.num_labels:
            return names
    return [f"style_{i}" for i in range(cfg.num_labels)]


def evaluate_random_baseline(cfg: TrainingConfig, dataset_dict: DatasetDict) -> Dict[str, Any]:
    y_true = np.array(dataset_dict["test"]["label"])
    rng = np.random.default_rng(cfg.seed)
    y_pred = rng.integers(low=0, high=cfg.num_labels, size=len(y_true))

    top1 = float((y_pred == y_true).mean())
    top5 = min(5 / cfg.num_labels, 1.0)
    macro_f1 = float(f1_score(y_true, y_pred, average="macro"))
    bal_acc = float(balanced_accuracy_score(y_true, y_pred))
    return {
        "method": "random_baseline",
        "top1": top1,
        "top5": float(top5),
        "macro_f1": macro_f1,
        "balanced_acc": bal_acc,
        "n_test": int(len(y_true)),
    }


def _build_eval_transform(processor: CLIPImageProcessor):
    _, eval_tf = get_transforms(processor)
    return eval_tf


def _batched_images(ds_split, start: int, end: int):
    imgs = ds_split[start:end]["image"]
    out = []
    for im in imgs:
        if isinstance(im, Image.Image):
            out.append(im.convert("RGB"))
        elif isinstance(im, dict) and "bytes" in im and im["bytes"] is not None:
            out.append(Image.open(io.BytesIO(im["bytes"])).convert("RGB"))
        else:
            out.append(im["image"].convert("RGB"))
    return out


def evaluate_zero_shot(cfg: TrainingConfig, dataset_dict: DatasetDict) -> Dict[str, Any]:
    device = get_device()
    model = CLIPModel.from_pretrained(cfg.model_id).to(device)
    processor = CLIPProcessor.from_pretrained(cfg.model_id)
    model.eval()

    style_names = _get_style_names(dataset_dict, cfg)
    prompts = [cfg.zero_shot_prompt_template.format(s) for s in style_names]

    with torch.no_grad():
        text_inputs = processor(text=prompts, return_tensors="pt", padding=True, truncation=True)
        text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
        text_emb = clip_text_features_compat(model, **text_inputs)
        text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)

    test = dataset_dict["test"]
    y_true = np.array(test["label"])
    logits_list = []
    bs = cfg.eval_batch_size_per_device
    for i in tqdm(range(0, len(test), bs), desc="zero-shot"):
        batch = test[i:i+bs]
        images = [img.convert("RGB") if isinstance(img, Image.Image) else img["image"].convert("RGB") for img in batch["image"]]
        with torch.no_grad():
            inputs = processor(images=images, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            image_emb = clip_image_features_compat(model, inputs["pixel_values"])
            image_emb = image_emb / image_emb.norm(dim=-1, keepdim=True)
            logits = image_emb @ text_emb.T
            logits_list.append(logits.detach().float().cpu().numpy())

    logits_np = np.concatenate(logits_list, axis=0)
    m = compute_metrics_from_logits(logits_np, y_true)
    return {
        "method": "clip_zero_shot",
        "top1": m["top1"],
        "top5": m["top5"],
        "macro_f1": m["macro_f1"],
        "balanced_acc": m["balanced_acc"],
        "n_test": int(len(y_true)),
    }


def smoke_test_zero_shot(cfg: TrainingConfig, dataset_dict: DatasetDict, n_images: int = 16) -> None:
    device = get_device()
    model = CLIPModel.from_pretrained(cfg.model_id).to(device)
    processor = CLIPProcessor.from_pretrained(cfg.model_id)
    model.eval()

    style_names = _get_style_names(dataset_dict, cfg)
    prompts = [cfg.zero_shot_prompt_template.format(s) for s in style_names]

    with torch.no_grad():
        text_inputs = processor(text=prompts, return_tensors="pt", padding=True, truncation=True)
        text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
        text_emb = clip_text_features_compat(model, **text_inputs)
        text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)

    test = dataset_dict["test"]
    n = min(n_images, len(test))
    batch = test.select(range(n))
    images = [img.convert("RGB") if isinstance(img, Image.Image) else img["image"].convert("RGB") for img in batch["image"]]

    with torch.no_grad():
        inputs = processor(images=images, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        image_emb = clip_image_features_compat(model, inputs["pixel_values"])
        image_emb = image_emb / image_emb.norm(dim=-1, keepdim=True)
        logits = image_emb @ text_emb.T

    assert logits.shape == (n, cfg.num_labels), f"Unexpected logits shape: {logits.shape}"
    assert torch.isfinite(logits).all(), "Non-finite values in zero-shot logits"
    print(f"Zero-shot smoke test passed. logits shape = {tuple(logits.shape)}")


def _extract_embeddings(cfg: TrainingConfig, ds_split, model: CLIPModel, processor: CLIPProcessor, device: torch.device):
    model.eval()
    ys = np.array(ds_split["label"])
    feats = []
    bs = cfg.eval_batch_size_per_device
    for i in tqdm(range(0, len(ds_split), bs), desc="embeddings"):
        batch = ds_split[i:i+bs]
        images = [img.convert("RGB") if isinstance(img, Image.Image) else img["image"].convert("RGB") for img in batch["image"]]
        with torch.no_grad():
            inputs = processor(images=images, return_tensors="pt")
            pixel_values = inputs["pixel_values"].to(device)
            emb = clip_image_features_compat(model, pixel_values)
            emb = emb / emb.norm(dim=-1, keepdim=True)
            feats.append(emb.detach().float().cpu().numpy())
    return np.concatenate(feats, axis=0), ys


def evaluate_linear_probe(cfg: TrainingConfig, dataset_dict: DatasetDict) -> Dict[str, Any]:
    device = get_device()
    clip_model = CLIPModel.from_pretrained(cfg.model_id).to(device)
    processor = CLIPProcessor.from_pretrained(cfg.model_id)

    x_train, y_train = _extract_embeddings(cfg, dataset_dict["train"], clip_model, processor, device)
    x_test, y_test = _extract_embeddings(cfg, dataset_dict["test"], clip_model, processor, device)

    probe = LogisticRegression(
        max_iter=500,
        multi_class="multinomial",
        class_weight="balanced",
        n_jobs=-1,
        random_state=cfg.seed,
    )
    probe.fit(x_train, y_train)
    logits = probe.predict_proba(x_test)
    m = compute_metrics_from_logits(logits, y_test)

    return {
        "method": "linear_probe",
        "top1": m["top1"],
        "top5": m["top5"],
        "macro_f1": m["macro_f1"],
        "balanced_acc": m["balanced_acc"],
        "n_test": int(len(y_test)),
    }


def evaluate_finetuned(cfg: TrainingConfig, dataset_dict: DatasetDict, checkpoint_path: str) -> Dict[str, Any]:
    device = get_device()
    amp_dtype = get_autocast_dtype(cfg)

    _, _, _, test_ds = build_torch_splits(dataset_dict, cfg)
    test_loader = _make_dataloader(test_ds, cfg, "test", 1, 0)

    model = build_model(cfg, dataset_dict).to(device)
    _ = _load_checkpoint(Path(checkpoint_path), model, map_location=device)
    test_metrics, logits_np, labels_np = evaluate_model(model, test_loader, device, amp_dtype)

    cm_path = Path(cfg.output_dir) / "confusion_matrix_finetuned_eval.npy"
    if cfg.save_confusion_matrix:
        cm = confusion_matrix(labels_np, logits_np.argmax(axis=1), labels=np.arange(cfg.num_labels))
        np.save(cm_path, cm)

    return {
        "method": "finetuned_clip",
        "top1": test_metrics["top1"],
        "top5": test_metrics["top5"],
        "macro_f1": test_metrics["macro_f1"],
        "balanced_acc": test_metrics["balanced_acc"],
        "n_test": int(len(labels_np)),
    }


def evaluate_all_methods(cfg: TrainingConfig, dataset_dict: DatasetDict, finetuned_checkpoint_path: str) -> pd.DataFrame:
    rows = []
    rows.append(evaluate_random_baseline(cfg, dataset_dict))
    rows.append(evaluate_zero_shot(cfg, dataset_dict))
    rows.append(evaluate_linear_probe(cfg, dataset_dict))
    rows.append(evaluate_finetuned(cfg, dataset_dict, finetuned_checkpoint_path))

    df = pd.DataFrame(rows)

    required = ["method", "top1", "top5", "macro_f1", "balanced_acc", "n_test"]
    for c in required:
        assert c in df.columns, f"Missing required metric key: {c}"
    assert set(df["method"].tolist()) == {"random_baseline", "clip_zero_shot", "linear_probe", "finetuned_clip"}
    assert not df[required].isnull().any().any(), "Comparison table has null values"

    if is_main_process():
        df.to_csv(Path(cfg.output_dir) / "comparison_metrics.csv", index=False)
        with open(Path(cfg.output_dir) / "comparison_metrics.json", "w", encoding="utf-8") as f:
            json.dump(rows, f, indent=2)
    return df


In [10]:
def run_smoke_test(cfg: TrainingConfig, dataset_dict: DatasetDict) -> None:
    cfg_smoke = copy.deepcopy(cfg)
    cfg_smoke.head_only_epochs = 1
    cfg_smoke.partial_unfreeze_epochs = 0
    cfg_smoke.early_stopping_patience = 1
    cfg_smoke.output_dir = str(Path(cfg.output_dir) / "smoke")
    cfg_smoke.checkpoint_dir = str(Path(cfg_smoke.output_dir) / "checkpoints")
    ensure_output_dirs(cfg_smoke)

    subset = DatasetDict(
        train=dataset_dict["train"].shuffle(seed=cfg.seed).select(range(min(cfg.smoke_subset_size, len(dataset_dict["train"])))),
        validation=dataset_dict["validation"].shuffle(seed=cfg.seed).select(range(min(cfg.smoke_subset_size // 4, len(dataset_dict["validation"])))),
        test=dataset_dict["test"].shuffle(seed=cfg.seed).select(range(min(cfg.smoke_subset_size // 4, len(dataset_dict["test"])))),
    )
    _ = train_finetune(cfg_smoke, subset)
    print("Smoke test completed.")


In [11]:
# Execution block
# For multi-GPU, launch notebook kernel via torchrun context or run equivalent script cells with env:
# torchrun --nproc_per_node=<N> your_script.py

ensure_output_dirs(config)
set_global_seed(config.seed)

if is_main_process():
    with open(Path(config.output_dir) / "config.json", "w", encoding="utf-8") as f:
        json.dump(asdict(config), f, indent=2)

dataset_dict = prepare_dataset(config)

# Optional quick check:
# run_smoke_test(config, dataset_dict)

best_ckpt = train_finetune(config, dataset_dict)
if is_main_process():
    summary = evaluate_all_methods(config, dataset_dict, best_ckpt)
    with open(Path(config.output_dir) / "best_checkpoint.txt", "w", encoding="utf-8") as f:
        f.write(best_ckpt)
    print("\nFinal comparison table:")
    display(summary)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Deduplicated by image_id before splitting/checks. Removed: {'train': 348}
Prepared dataset with split sizes: {'train': 82321, 'validation': 10290, 'test': 10291}


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Stage: head | trainable params: 70,281


...

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Final comparison table:


,method,top1,top5,macro_f1,balanced_acc,n_test
0,random_baseline,0.006705,0.036496,0.003302,0.003678,10291
1,clip_zero_shot,0.082402,0.279662,0.059319,0.114575,10291
2,linear_probe,0.359926,0.759596,0.296852,0.565469,10291
3,finetuned_clip,0.395103,0.786027,0.271845,0.346086,10291
